# Aula 08 - Notebook: Sistemas Especialistas — Base de Conhecimento e Regras de Diagnóstico

Este notebook apresenta o desenvolvimento de um **Sistema Especialista Baseado em Regras** (*Rule-Based Expert System*) para a **Linha Automatizada de Envasamento e Tampamento de Bebidas (Grupo 3)**.

## 1. Fundamentos Matemáticos: Arquitetura de Sistemas Especialistas (RBS)

No contexto da automação e segurança funcional de processos (normas IEC 61508 / IEC 61511), a detecção e diagnóstico rápido de falhas baseia-se na formalização de uma base de conhecimento. Um **Sistema Baseado em Regras** (RBS) é definido pela tripla:

$$\langle \mathcal{F}, \mathcal{R}, \mathcal{E} \rangle$$

Onde:
1. **$\mathcal{F}$ (Base de Fatos):** Conjunto finito de proposições lógicas ativas num instante de tempo $t$, representando o estado atual da planta de envasamento:
   $$\mathcal{F}(t) = \{f_1, f_2, \dots, f_m\} \subseteq \mathcal{U}_{\text{fatos}}$$
2. **$\mathcal{R}$ (Base de Conhecimento / Regras):** Conjunto de regras lógicas de produção expressas na forma de **Cláusulas de Horn Definidas**:
   $$R_i: \quad \text{SE } (A_{i,1} \land A_{i,2} \land \dots \land A_{i,k}) \quad \text{ENTÃO } \quad C_i$$
   Cada antecedente $A_{i,j}$ é um fato associado à leitura de um sensor (como `p_max1`, `y_bomba`) e o consequente $C_i$ representa um fato derivado (como `CAVITACAO_BOMBA_BC1`).
3. **$\mathcal{E}$ (Estratégia de Resolução e Conflito):** Regras de arbitragem utilizadas pelo motor de inferência para priorização e disparo de alarmes baseadas na severidade do risco operacional e de segurança.

## 2. Modelagem das Classes: Fato, RegraDiagnostico e BaseConhecimentoSCADA

Implementamos a arquitetura do sistema especialista utilizando programação orientada a objetos em Python. As classes essenciais criadas são:
- `Fato`: Representa proposições lógicas vindas de telemetria ou inferidas, contendo o valor booleano, descrição, fonte do fato e carimbo de tempo (*timestamp*).
- `RegraDiagnostico`: Contém a identificação da regra, o conjunto de antecedentes, o consequente derivado, a descrição do diagnóstico, a severidade operacional (Crítica, Alta, Média, Baixa), a prioridade numérica (1 a 10), o tempo de resposta máximo aceitável e o Procedimento Operacional Padrão (POP) a ser seguido pelo operador.
- `BaseConhecimentoSCADA`: Gerenciador central responsável por indexar as regras de diagnóstico, indexar os antecedentes para busca rápida, exportar a base de regras no formato de catálogo tabular e validar a integridade da base por meio de verificações de consistência e detecção de redundâncias.

In [1]:
from dataclasses import dataclass, field
from typing import List, Set, Dict, Any, Optional
import time

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata uma lista de dicionários em uma tabela ASCII legível para exibição técnica."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR"  # SENSOR ou INFERIDO
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str             # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int             # 1 a 10 (10 = máxima urgência)
    tempo_resposta_max_s: float
    procedimento_pop: str

class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, pop: str = "Verificar malha"
    ):
        """Cadastra uma nova regra de diagnóstico e atualiza o índice invertido."""
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)

        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

    def obter_regras_por_fato(self, fato_nome: str) -> List[RegraDiagnostico]:
        """Busca rapidamente regras que dependem de um determinado fato ativado."""
        return self._indice_antecedentes.get(fato_nome, [])

    def verificar_consistencia(self) -> List[str]:
        """Verifica se existem regras contraditórias (mesmos antecedentes mas consequentes diferentes)."""
        inconsistencias = []
        grupos: Dict[tuple, List[RegraDiagnostico]] = {}
        for r in self.regras:
            chave = tuple(sorted(r.antecedentes))
            if chave not in grupos:
                grupos[chave] = []
            grupos[chave].append(r)

        for ant, regras_grupo in grupos.items():
            if len(regras_grupo) > 1:
                for i in range(len(regras_grupo)):
                    for j in range(i + 1, len(regras_grupo)):
                        r1 = regras_grupo[i]
                        r2 = regras_grupo[j]
                        if r1.consequente != r2.consequente:
                            inconsistencias.append(
                                f"Contradição detectada: Regras {r1.id_regra} e {r2.id_regra} "
                                f"possuem os mesmos antecedentes {list(ant)} mas geram consequentes diferentes: "
                                f"'{r1.consequente}' vs '{r2.consequente}'."
                            )
        return inconsistencias

    def verificar_redundancias(self) -> List[str]:
        """Verifica se existem regras redundantes (mesmos antecedentes e mesmo consequente)."""
        redundancias = []
        grupos: Dict[tuple, List[RegraDiagnostico]] = {}
        for r in self.regras:
            chave = tuple(sorted(r.antecedentes))
            if chave not in grupos:
                grupos[chave] = []
            grupos[chave].append(r)

        for ant, regras_grupo in grupos.items():
            if len(regras_grupo) > 1:
                consequentes = {}
                for r in regras_grupo:
                    if r.consequente in consequentes:
                        r_orig = consequentes[r.consequente]
                        redundancias.append(
                            f"Redundância detectada: Regras {r.id_regra} e {r_orig.id_regra} "
                            f"possuem os mesmos antecedentes {list(ant)} e o mesmo consequente '{r.consequente}'."
                        )
                    else:
                        consequentes[r.consequente] = r
        return redundancias

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        """Exporta o catálogo de regras ordenado por prioridade decrescente."""
        catalogo = []
        for r in sorted(self.regras, key=lambda x: x.prioridade, reverse=True):
            catalogo.append({
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Consequente)": r.consequente,
                "Diagnóstico": r.descricao_diagnostico,
                "POP": r.procedimento_pop
            })
        return catalogo

print("[OK] Estrutura orientada a objetos da Base de Conhecimento inicializada com sucesso!")

[OK] Estrutura orientada a objetos da Base de Conhecimento inicializada com sucesso!


## 3. Cadastro e Integração do Catálogo de Regras do Grupo 3

Agora cadastramos as 6 regras estruturadas para o nosso sistema da **Linha de Envasamento de Bebidas (Grupo 3)**. A tabela a seguir relaciona as regras cadastradas:

| ID | Antecedentes | Consequente | Diagnóstico de Causa-Raiz | Severidade / Ação |
| :--- | :--- | :--- | :--- | :--- |
| **R-01** | `p_max1` $\land$ `q_max1` | `SOBRECARGA_LINHA_ALIMENTACAO` | Sobrecarga de Pressão/Vazão na Entrada da Linha | **CRÍTICA:** Desarmar bomba BC1 e fechar VS1 |
| **R-02** | `SOBRECARGA_LINHA_ALIMENTACAO` $\land$ `y_valv1` | `TRIP_BLOQUEIO_EMERGENCIA` | Falha de Alívio com Válvula Principal VS1 Aberta | **CRÍTICA:** Fechar VS1 e desligar BC1 via contator |
| **R-03** | `p_min1` $\land$ `y_bomba` | `CAVITACAO_BOMBA_BC1` | Risco de Cavitação e Destruição Mecânica na Bomba BC1 | **ALTA:** Desligar bomba BC1 e checar nível de TS1 |
| **R-04** | `p_max2` | `SOBREPRESSAO_ACUMULADOR_AS1` | Pressão Elevada de Projeto no Acumulador de Suprimento AS1 | **CRÍTICA:** Aliviar pressão e cortar fluxo para AS1 |
| **R-05** | `q_min2` $\land$ `y_valv2` | `OBSTRUCAO_BICO_ENVASE` | Bloqueio ou Entupimento Mecânico no Bico de Envase VS2 | **ALTA:** Parar esteira RC1 e realizar retrolavagem |
| **R-06** | `p_max2` $\land$ `y_valv3` | `SOBREPRESSAO_SISTEMA_CAPPING` | Pressão Pneumática Excessiva no Atuador de Tampamento AC1 | **CRÍTICA:** Fechar válvula VS3 de capping e aliviar ar |

In [2]:
bc = BaseConhecimentoSCADA()

# Cadastro das regras adaptadas à Linha de Envase de Bebidas (Grupo 3)
bc.adicionar_regra(
    id_regra="R-01",
    antecedentes=["p_max1", "q_max1"],
    consequente="SOBRECARGA_LINHA_ALIMENTACAO",
    descricao="Sobrecarga de Pressão e Vazão na Linha de Alimentação",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=1.0,
    pop="POP-SIS-01: Cortar alimentação, parar bomba BC1 e fechar válvula VS1"
)

bc.adicionar_regra(
    id_regra="R-02",
    antecedentes=["SOBRECARGA_LINHA_ALIMENTACAO", "y_valv1"],
    consequente="TRIP_BLOQUEIO_EMERGENCIA",
    descricao="Falha de Alívio com Válvula Principal VS1 Aberta sob Sobrecarga",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.5,
    pop="POP-SIS-02: Interromper contator da bomba BC1 e forçar fechamento de VS1 via PLC"
)

bc.adicionar_regra(
    id_regra="R-03",
    antecedentes=["p_min1", "y_bomba"],
    consequente="CAVITACAO_BOMBA_BC1",
    descricao="Risco Crítico de Cavitação e Falha Mecânica na Bomba BC1",
    severidade="ALTA",
    prioridade=8,
    tempo_max_s=2.0,
    pop="POP-MA-04: Desligar bomba BC1 e verificar nível do tanque de suprimento TS1"
)

bc.adicionar_regra(
    id_regra="R-04",
    antecedentes=["p_max2"],
    consequente="SOBREPRESSAO_ACUMULADOR_AS1",
    descricao="Sobrepressão Acima do Limite de Projeto no Acumulador AS1",
    severidade="CRÍTICA",
    prioridade=9,
    tempo_max_s=1.0,
    pop="POP-SST-08: Acionar alívio pneumático de emergência e interromper fluxo para AS1"
)

bc.adicionar_regra(
    id_regra="R-05",
    antecedentes=["q_min2", "y_valv2"],
    consequente="OBSTRUCAO_BICO_ENVASE",
    descricao="Bloqueio ou Entupimento Mecânico no Bico de Envase VS2",
    severidade="ALTA",
    prioridade=7,
    tempo_max_s=3.0,
    pop="POP-SEC-02: Parar esteira RC1, isolar ramal de envase e realizar retrolavagem"
)

bc.adicionar_regra(
    id_regra="R-06",
    antecedentes=["p_max2", "y_valv3"],
    consequente="SOBREPRESSAO_SISTEMA_CAPPING",
    descricao="Pressão Pneumática Excessiva no Atuador de Capping AC1",
    severidade="CRÍTICA",
    prioridade=9,
    tempo_max_s=2.0,
    pop="POP-CRIO-01: Fechar válvula de capping VS3 e aliviar pressão residual"
)

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (GRUPO 3) ===\n")
print(formatar_tabela(bc.exportar_catalogo()))

=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (GRUPO 3) ===

ID   | Prioridade | Severidade | SE (Antecedentes)                        | ENTÃO (Consequente)          | Diagnóstico                                                     | POP                                                                             
-----+------------+------------+------------------------------------------+------------------------------+-----------------------------------------------------------------+---------------------------------------------------------------------------------
R-01 | 10         | CRÍTICA    | p_max1 AND q_max1                        | SOBRECARGA_LINHA_ALIMENTACAO | Sobrecarga de Pressão e Vazão na Linha de Alimentação           | POP-SIS-01: Cortar alimentação, parar bomba BC1 e fechar válvula VS1            
R-02 | 10         | CRÍTICA    | SOBRECARGA_LINHA_ALIMENTACAO AND y_valv1 | TRIP_BLOQUEIO_EMERGENCIA     | Falha de Alívio com Válvula Principal VS1 Aberta sob Sobr

## 4. Testes de Validação e Verificação Semântica da Base

Nesta seção, validamos a integridade da base de conhecimento por meio de asserções estáticas, e testamos os mecanismos algorítmicos de detecção de anomalias (contradições e redundâncias).

In [3]:
# 1. Testes Estáticos Básicos
assert len(bc.regras) == 6, "Erro: A base deve conter exatamente 6 regras de diagnóstico."

# Verificar o índice invertido de busca por fatos
regras_p_max1 = bc.obter_regras_por_fato("p_max1")
assert any(r.id_regra == "R-01" for r in regras_p_max1), "Erro: R-01 deve ser recuperada a partir do fato p_max1."

# Garantir que a base cadastrada está consistente e sem redundâncias inicialmente
assert len(bc.verificar_consistencia()) == 0, "Erro: A base inicial contém contradições inesperadas."
assert len(bc.verificar_redundancias()) == 0, "Erro: A base inicial contém redundâncias inesperadas."
print("[OK] Base de regras principal está 100% consistente e livre de redundâncias!")


# 2. Simulação de Anomalias (Inserção controlada de inconsistências)
print("\n--- Iniciando Testes Dinâmicos de Detecção de Anomalias ---")

# Criando uma instância temporária para testes destrutivos
bc_teste = BaseConhecimentoSCADA()
for r in bc.regras:
    bc_teste.adicionar_regra(
        r.id_regra, list(r.antecedentes), r.consequente,
        r.descricao_diagnostico, r.severidade, r.prioridade,
        r.tempo_resposta_max_s, r.procedimento_pop
    )

# Injetando inconsistência / contradição:
# Mesma condição de R-01 (p_max1 e q_max1), mas com consequente diferente
print("\n[Ação] Injetando regra contraditória R-XX-CONTRADITORIA...")
bc_teste.adicionar_regra(
    id_regra="R-XX-CONTRADITORIA",
    antecedentes=["p_max1", "q_max1"],
    consequente="OPERACAO_NORMAL_LINHA", # Contradiz R-01 que infere SOBRECARGA_LINHA_ALIMENTACAO
    descricao="Indicação falsa de operação normal sob alta vazão e pressão",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=1.0,
    pop="POP-IGNORAR"
)

inconsistencias = bc_teste.verificar_consistencia()
print(f"Inconsistências encontradas (esperado > 0): {len(inconsistencias)}")
for inc in inconsistencias:
    print(f"  - ALERT: {inc}")
assert len(inconsistencias) > 0, "Erro: O algoritmo falhou ao detectar a contradição entre R-01 e R-XX-CONTRADITORIA."


# Injetando redundância:
# Mesma condição de R-03 (p_min1 e y_bomba) e mesmo consequente (CAVITACAO_BOMBA_BC1)
print("\n[Ação] Injetando regra redundante R-YY-REDUNDANTE...")
bc_teste.adicionar_regra(
    id_regra="R-YY-REDUNDANTE",
    antecedentes=["p_min1", "y_bomba"],
    consequente="CAVITACAO_BOMBA_BC1",
    descricao="Mesma detecção de cavitação que R-03",
    severidade="ALTA",
    prioridade=8,
    tempo_max_s=2.0,
    pop="POP-MA-04"
)

redundancias = bc_teste.verificar_redundancias()
print(f"Redundâncias encontradas (esperado > 0): {len(redundancias)}")
for red in redundancias:
    print(f"  - ALERT: {red}")
assert len(redundancias) > 0, "Erro: O algoritmo falhou ao detectar a redundância entre R-03 e R-YY-REDUNDANTE."

print("\n[OK] Todos os testes de validação semântica e funcional passaram com sucesso!")

[OK] Base de regras principal está 100% consistente e livre de redundâncias!

--- Iniciando Testes Dinâmicos de Detecção de Anomalias ---

[Ação] Injetando regra contraditória R-XX-CONTRADITORIA...
Inconsistências encontradas (esperado > 0): 1
  - ALERT: Contradição detectada: Regras R-01 e R-XX-CONTRADITORIA possuem os mesmos antecedentes ['p_max1', 'q_max1'] mas geram consequentes diferentes: 'SOBRECARGA_LINHA_ALIMENTACAO' vs 'OPERACAO_NORMAL_LINHA'.

[Ação] Injetando regra redundante R-YY-REDUNDANTE...
Redundâncias encontradas (esperado > 0): 1
  - ALERT: Redundância detectada: Regras R-YY-REDUNDANTE e R-03 possuem os mesmos antecedentes ['p_min1', 'y_bomba'] e o mesmo consequente 'CAVITACAO_BOMBA_BC1'.

[OK] Todos os testes de validação semântica e funcional passaram com sucesso!
